# 04. Smoothing ablation

Этот notebook читает сохраненный smoothing ablation из `results/charades_sta_smoothing_1000/summary.csv`. Он использует ту же фиксированную 1,000-query Charades-STA subset, что и main CLIP sweep, и не запускает CLIP inference.


In [3]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent

smoothing = pd.read_csv(ROOT / 'results/charades_sta_smoothing_1000/summary.csv')
cols = ['config_name', 'window_size', 'stride', 'aggregation', 'smoothing', 'evaluated_queries', 'R@1_IoU_0.3', 'R@1_IoU_0.5', 'R@1_IoU_0.7', 'mIoU', 'inference_time_sec']
smoothing[cols]


,config_name,window_size,stride,aggregation,smoothing,evaluated_queries,R@1_IoU_0.3,R@1_IoU_0.5,R@1_IoU_0.7,mIoU,inference_time_sec
0,clip_w8_s4_mean_none,8.0,4.0,mean,none,1000,0.509,0.393,0.181,0.347777,17.297376
1,clip_w8_s4_mean_moving_average_3,8.0,4.0,mean,moving_average_3,1000,0.511,0.398,0.183,0.348520,15.721828
2,clip_w8_s4_mean_moving_average_5,8.0,4.0,mean,moving_average_5,1000,0.509,0.398,0.190,0.349421,15.789070
3,clip_w16_s8_mean_none,16.0,8.0,mean,none,1000,0.625,0.260,0.096,0.349690,14.787219
4,clip_w16_s8_mean_moving_average_3,16.0,8.0,mean,moving_average_3,1000,0.620,0.257,0.098,0.349082,15.369382
5,clip_w16_s8_mean_moving_average_5,16.0,8.0,mean,moving_average_5,1000,0.623,0.254,0.098,0.348085,15.995661


In [4]:
base_cols = ['window_size', 'stride', 'aggregation']
summary = []
for keys, group in smoothing.groupby(base_cols):
    base = dict(zip(base_cols, keys))
    for metric in ['R@1_IoU_0.7', 'mIoU']:
        row = group.loc[group[metric].idxmax()]
        summary.append({**base, 'metric': metric, 'best_smoothing': row['smoothing'], 'value': row[metric]})
pd.DataFrame(summary)


,window_size,stride,aggregation,metric,best_smoothing,value
0,8.0,4.0,mean,R@1_IoU_0.7,moving_average_5,0.190000
1,8.0,4.0,mean,mIoU,moving_average_5,0.349421
2,16.0,8.0,mean,R@1_IoU_0.7,moving_average_3,0.098000
3,16.0,8.0,mean,mIoU,none,0.349690


Вывод: smoothing является небольшой ablation, а не основным результатом. Он немного улучшает strict localization для более короткой настройки `w8/s4/mean`, но не дает устойчивого улучшения для всех configurations.


Примечание о воспроизводимости: heavy experiment runner - `scripts/run_smoothing_sweep.py`; этот notebook читает только сохраненные public results.
